In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — INSTALL, IMPORTS, CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, time, copy, warnings, logging, shutil
import numpy as np
import scipy.io
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt
import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MultipleLocator
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.max_open_warning"] = 0

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

# ═══════════════════════════════════════════════════════════════════════════════
# LOCAL PATH CONFIGURATION — SET YOUR DATA DIRECTORY HERE
# ═══════════════════════════════════════════════════════════════════════════════
BASE_DATA_DIR = r"C:\Users\reonp\.vscode\Codes\MNNIT"
# ═══════════════════════════════════════════════════════════════════════════════

DATA_DIR  = BASE_DATA_DIR
SAVE_DIR  = BASE_DATA_DIR
DRIVE_DIR = os.path.join(BASE_DATA_DIR, "saved")
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Platform : Local (VS Code Jupyter)")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"SAVE_DIR  : {SAVE_DIR}")
print(f"DRIVE_DIR : {DRIVE_DIR}")

def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        name   = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU detected : {name} ({mem_gb:.1f} GB)")
    else:
        device = torch.device("cpu")
        print("No GPU — running on CPU")
    return device

DEVICE = get_device()

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)

PREPROCESS_CFG = {
    "sfreq"              : 200.0,
    "target_sfreq"       : 200.0,
    "bandpass_low"       : 0.5,
    "bandpass_high"      : 45.0,
    "filter_order"       : 4,
    "notch_freq"         : 50.0,
    "notch_quality"      : 30.0,
    "artifact_std_thresh": 3.0,
    "subject_clip_sigma" : 4.0,
}

SEGMENT_CFG = {
    "expected_time_steps" : 384,
}

MODEL_CFG = {
    "latent_dim"   : 128,
    "fno_width"    : 512,
    "n_modes"      : 32,
    "n_fno_layers" : 3,
    "base_filters" : 64,
    "hidden_dim"   : 256,
    "num_classes"  : 1,
    "dropout_rate" : 0.5,
}

TRAIN_CFG = {
    "lr"           : 1e-4,
    "weight_decay" : 1e-4,
    "lambda1"      : 0.85,
    "lambda2"      : 0.1,
    "seed_epochs"  : 50,
    "sadt_epochs"  : 60,
    "batch_size"   : 16,
    "noise_std"    : 0.05,
    "num_workers"  : 0,
    "patience"     : 12,
    "min_delta"    : 1e-4,
    "seed"         : 42,
}

INFERENCE_CFG = {
    "threshold" : 0.5,
}

def save_to_drive(filename):
    src = os.path.join(SAVE_DIR, filename)
    dst = os.path.join(DRIVE_DIR, filename)
    if os.path.isfile(src):
        shutil.copy(src, dst)
        print(f"  Saved to drive : {filename}")

def restore_from_drive(filename):
    src = os.path.join(DRIVE_DIR, filename)
    dst = os.path.join(SAVE_DIR, filename)
    if os.path.isfile(src):
        shutil.copy(src, dst)
        print(f"  Restored from drive : {filename}")
        return True
    return False

def check_file(filename):
    path = os.path.join(SAVE_DIR, filename)
    if os.path.isfile(path):
        return True
    return restore_from_drive(filename)

print("\nCell 1 Complete — Setup & Configuration loaded.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — DATA LOADING & SMART PREPROCESSING
# ═══════════════════════════════════════════════════════════════════════════════

def find_file(name):
    direct_path = os.path.join(DATA_DIR, name)
    if os.path.isfile(direct_path):
        return direct_path
    
    for root, dirs, files_list in os.walk(DATA_DIR):
        if name in files_list:
            return os.path.join(root, name)
    
    for search_dir in [SAVE_DIR, DRIVE_DIR]:
        for root, dirs, files_list in os.walk(search_dir):
            if name in files_list:
                return os.path.join(root, name)
    
    return None

def load_seed_vig():
    path = find_file("SEED_VIG.mat")
    if path is None:
        raise FileNotFoundError(f"SEED_VIG.mat not found in {DATA_DIR}")
    print(f"  Loading SEED-VIG from: {path}")
    mat      = scipy.io.loadmat(path)
    eeg      = mat["EEGsample"].astype(np.float64)
    labels   = mat["substate"].squeeze()
    subindex = mat["subindex"].squeeze() if "subindex" in mat else None
    print(f"  EEG: {eeg.shape} | Labels: {labels.shape}")
    return eeg, labels, subindex

def load_sadt():
    path = find_file("dataset.mat")
    if path is None:
        raise FileNotFoundError(f"dataset.mat not found in {DATA_DIR}")
    print(f"  Loading SADT from: {path}")
    mat      = scipy.io.loadmat(path)
    eeg      = mat["EEGsample"].astype(np.float64)
    labels   = mat["substate"].squeeze()
    subindex = mat["subindex"].squeeze() if "subindex" in mat else None
    print(f"  EEG: {eeg.shape} | Labels: {labels.shape}")
    return eeg, labels, subindex

def detect_bad_channels(data, threshold_multiplier=3.0):
    n_samples, n_channels, n_times = data.shape
    channel_stds = np.array([data[:, c, :].std() for c in range(n_channels)])
    median_std   = np.median(channel_stds)
    bad_channels = []
    print(f"\n  Channel Std Analysis (median std = {median_std:.4f}):")
    print(f"  {'CH':<6} {'Std':>10} {'Ratio':>8} {'Status':>10}")
    print(f"  {'-'*40}")
    for c in range(n_channels):
        ratio  = channel_stds[c] / median_std
        status = "BAD" if ratio > threshold_multiplier else "OK"
        if ratio > threshold_multiplier:
            bad_channels.append(c)
        print(f"  CH{c+1:<4} {channel_stds[c]:>10.4f} {ratio:>8.2f}x {status:>10}")
    return bad_channels

def remove_channels(data, bad_channels, dataset_name="Dataset"):
    if len(bad_channels) == 0:
        print(f"  {dataset_name}: No bad channels to remove.")
        return data
    good_mask    = [c for c in range(data.shape[1]) if c not in bad_channels]
    data_cleaned = data[:, good_mask, :]
    removed_names = [f"CH{c+1}" for c in bad_channels]
    print(f"  {dataset_name}: Removed {removed_names}")
    print(f"     Shape: {data.shape} -> {data_cleaned.shape}")
    return data_cleaned

def clip_per_subject(data, subindex, sigma=4.0):
    if subindex is None:
        mean = data.mean()
        std  = data.std()
        lo, hi = mean - sigma * std, mean + sigma * std
        clipped = np.sum((data < lo) | (data > hi))
        data = np.clip(data, lo, hi)
        print(f"  Global clipping at +/-{sigma}sigma : {clipped} values clipped")
        return data
    data_clipped  = data.copy()
    total_clipped = 0
    unique_subjs  = np.unique(subindex)
    print(f"\n  Per-Subject Clipping (sigma = {sigma}):")
    print(f"  {'Subject':<10} {'Mean':>10} {'Std':>10} {'Clipped':>10}")
    print(f"  {'-'*45}")
    for sid in unique_subjs:
        mask      = subindex == sid
        subj_data = data_clipped[mask]
        mean      = subj_data.mean()
        std       = subj_data.std()
        lo, hi    = mean - sigma * std, mean + sigma * std
        clipped   = np.sum((subj_data < lo) | (subj_data > hi))
        total_clipped += clipped
        data_clipped[mask] = np.clip(subj_data, lo, hi)
        print(f"  S{sid:<9} {mean:>10.4f} {std:>10.4f} {clipped:>10}")
    print(f"\n  Total values clipped : {total_clipped}")
    return data_clipped

def normalize_per_subject(data, subindex, epsilon=1e-8):
    if subindex is None:
        for s in range(data.shape[0]):
            for c in range(data.shape[1]):
                mean = data[s, c, :].mean()
                std  = data[s, c, :].std()
                data[s, c, :] = (data[s, c, :] - mean) / (std + epsilon)
        return data
    data_norm    = data.copy()
    unique_subjs = np.unique(subindex)
    print(f"\n  Per-Subject Normalization:")
    print(f"  {'Subject':<10} {'Pre-Mean':>10} {'Pre-Std':>10} {'Post-Mean':>10} {'Post-Std':>10}")
    print(f"  {'-'*55}")
    for sid in unique_subjs:
        mask      = subindex == sid
        subj_data = data_norm[mask]
        pre_mean  = subj_data.mean()
        pre_std   = subj_data.std()
        for i in range(subj_data.shape[0]):
            for c in range(subj_data.shape[1]):
                signal = subj_data[i, c, :]
                mean   = signal.mean()
                std    = signal.std()
                subj_data[i, c, :] = (signal - mean) / (std + epsilon)
        data_norm[mask] = subj_data
        post_mean = data_norm[mask].mean()
        post_std  = data_norm[mask].std()
        print(f"  S{sid:<9} {pre_mean:>10.4f} {pre_std:>10.4f} {post_mean:>10.4f} {post_std:>10.4f}")
    return data_norm

def apply_bandpass(data, sfreq, lowcut=0.5, highcut=45.0, order=4):
    nyquist = sfreq / 2.0
    sos = butter(order, [lowcut / nyquist, highcut / nyquist], btype="band", output="sos")
    filtered = np.zeros_like(data)
    for s in range(data.shape[0]):
        for c in range(data.shape[1]):
            filtered[s, c, :] = sosfiltfilt(sos, data[s, c, :])
    return filtered

def apply_notch(data, sfreq, freq=50.0, quality=30.0):
    nyquist = sfreq / 2.0
    if freq >= nyquist:
        return data.copy()
    b, a = iirnotch(freq, quality, fs=sfreq)
    filtered = np.zeros_like(data)
    for s in range(data.shape[0]):
        for c in range(data.shape[1]):
            filtered[s, c, :] = filtfilt(b, a, data[s, c, :])
    return filtered

def preprocess_dataset(data, labels, subindex, dataset_name,
                       sfreq=200.0, artifact_thresh=3.0, clip_sigma=4.0):
    print(f"\n{'='*70}")
    print(f"  PREPROCESSING: {dataset_name}")
    print(f"  Input shape: {data.shape}")
    print(f"{'='*70}")

    print(f"\n[STEP 1] Detecting corrupted channels ...")
    bad_channels = detect_bad_channels(data, artifact_thresh)
    data = remove_channels(data, bad_channels, dataset_name)

    print(f"\n[STEP 2] Handling NaN values ...")
    nan_count = np.isnan(data).sum()
    if nan_count > 0:
        for s in range(data.shape[0]):
            for c in range(data.shape[1]):
                signal   = data[s, c, :]
                nan_mask = np.isnan(signal)
                if nan_mask.any():
                    ch_mean = np.nanmean(signal)
                    if np.isnan(ch_mean): ch_mean = 0.0
                    data[s, c, nan_mask] = ch_mean
        print(f"  Fixed {nan_count} NaN values")
    else:
        print(f"  No NaN values found")

    print(f"\n[STEP 3] Per-subject outlier clipping ...")
    data = clip_per_subject(data, subindex, sigma=clip_sigma)

    print(f"\n[STEP 4] Band-pass filtering (0.5-45 Hz) ...")
    data = apply_bandpass(data, sfreq)
    print(f"  Band-pass complete")

    print(f"\n[STEP 5] Notch filtering (50 Hz) ...")
    data = apply_notch(data, sfreq)
    print(f"  Notch complete")

    print(f"\n[STEP 6] Per-subject z-score normalization ...")
    data = normalize_per_subject(data, subindex)

    print(f"\n{'='*70}")
    print(f"  {dataset_name} PREPROCESSING COMPLETE")
    print(f"  Output shape : {data.shape}")
    print(f"  Channels     : {data.shape[1]} (removed {len(bad_channels)} bad)")
    print(f"  NaN          : {np.isnan(data).sum()}")
    print(f"  Inf          : {np.isinf(data).sum()}")
    print(f"  Mean         : {data.mean():.6f}")
    print(f"  Std          : {data.std():.6f}")
    print(f"{'='*70}")

    return data.astype(np.float64), bad_channels

all_exist = all([
    check_file("seed_clean_x.npy"),
    check_file("seed_clean_y.npy"),
    check_file("sadt_clean_x.npy"),
    check_file("sadt_clean_y.npy"),
    check_file("sadt_clean_subindex.npy"),
    check_file("channel_info.npy"),
])

if all_exist:
    print("Preprocessed files already exist. Skipping preprocessing.")
    channel_info  = np.load(os.path.join(SAVE_DIR, "channel_info.npy"), allow_pickle=True).item()
    SEED_CHANNELS = channel_info["seed_channels"]
    SADT_CHANNELS = channel_info["sadt_channels"]
    print(f"   SEED Channels : {SEED_CHANNELS}")
    print(f"   SADT Channels : {SADT_CHANNELS}")
else:
    print("Running full preprocessing pipeline ...")
    
    seed_eeg, seed_labels, seed_subindex = load_seed_vig()
    seed_eeg, seed_bad_ch = preprocess_dataset(
        seed_eeg, seed_labels, seed_subindex, "SEED-VIG",
        sfreq=PREPROCESS_CFG["sfreq"],
        artifact_thresh=PREPROCESS_CFG["artifact_std_thresh"],
        clip_sigma=PREPROCESS_CFG["subject_clip_sigma"]
    )
    SEED_CHANNELS = seed_eeg.shape[1]

    sadt_eeg, sadt_labels, sadt_subindex = load_sadt()
    sadt_eeg, sadt_bad_ch = preprocess_dataset(
        sadt_eeg, sadt_labels, sadt_subindex, "SADT",
        sfreq=PREPROCESS_CFG["sfreq"],
        artifact_thresh=PREPROCESS_CFG["artifact_std_thresh"],
        clip_sigma=PREPROCESS_CFG["subject_clip_sigma"]
    )
    SADT_CHANNELS = sadt_eeg.shape[1]

    np.save(os.path.join(SAVE_DIR, "seed_clean_x.npy"), seed_eeg)
    np.save(os.path.join(SAVE_DIR, "seed_clean_y.npy"), seed_labels)
    np.save(os.path.join(SAVE_DIR, "sadt_clean_x.npy"), sadt_eeg)
    np.save(os.path.join(SAVE_DIR, "sadt_clean_y.npy"), sadt_labels)
    np.save(os.path.join(SAVE_DIR, "sadt_clean_subindex.npy"), sadt_subindex)

    channel_info = {
        "seed_channels" : SEED_CHANNELS,
        "sadt_channels" : SADT_CHANNELS,
        "seed_bad_ch"   : seed_bad_ch,
        "sadt_bad_ch"   : sadt_bad_ch,
    }
    np.save(os.path.join(SAVE_DIR, "channel_info.npy"), channel_info)

    for f in ["seed_clean_x.npy", "seed_clean_y.npy",
              "sadt_clean_x.npy", "sadt_clean_y.npy",
              "sadt_clean_subindex.npy", "channel_info.npy"]:
        save_to_drive(f)

    print(f"\nPreprocessing complete & backed up.")
    print(f"   SEED Channels : {SEED_CHANNELS} (removed {len(seed_bad_ch)})")
    print(f"   SADT Channels : {SADT_CHANNELS} (removed {len(sadt_bad_ch)})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — SEGMENTATION & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

seg_exist = all([
    check_file("seed_ready_X.npy"),
    check_file("seed_ready_Y.npy"),
    check_file("sadt_ready_X.npy"),
    check_file("sadt_ready_Y.npy"),
    check_file("sadt_subject_map.npy"),
])

if seg_exist:
    print("Segmented files already exist. Skipping segmentation.")
else:
    print("Running segmentation ...")

    seed_x   = np.load(os.path.join(SAVE_DIR, "seed_clean_x.npy"))
    seed_y   = np.load(os.path.join(SAVE_DIR, "seed_clean_y.npy"))
    sadt_x   = np.load(os.path.join(SAVE_DIR, "sadt_clean_x.npy"))
    sadt_y   = np.load(os.path.join(SAVE_DIR, "sadt_clean_y.npy"))
    sadt_sub = np.load(os.path.join(SAVE_DIR, "sadt_clean_subindex.npy"))

    print(f"  SEED-VIG : {seed_x.shape}")
    print(f"  SADT     : {sadt_x.shape}")

    assert seed_x.shape[2] == 384
    assert sadt_x.shape[2] == 384
    print(f"  Time steps validated (384)")

    seed_x_4d = seed_x[:, :, :, np.newaxis]
    sadt_x_4d = sadt_x[:, :, :, np.newaxis]

    seed_y_int = seed_y.astype(np.int64)
    sadt_y_int = sadt_y.astype(np.int64)
    sadt_map   = sadt_sub.astype(np.int64)

    np.save(os.path.join(SAVE_DIR, "seed_ready_X.npy"), seed_x_4d)
    np.save(os.path.join(SAVE_DIR, "seed_ready_Y.npy"), seed_y_int)
    np.save(os.path.join(SAVE_DIR, "sadt_ready_X.npy"), sadt_x_4d)
    np.save(os.path.join(SAVE_DIR, "sadt_ready_Y.npy"), sadt_y_int)
    np.save(os.path.join(SAVE_DIR, "sadt_subject_map.npy"), sadt_map)

    for f in ["seed_ready_X.npy", "seed_ready_Y.npy",
              "sadt_ready_X.npy", "sadt_ready_Y.npy",
              "sadt_subject_map.npy"]:
        save_to_drive(f)

    unique_subjs = np.unique(sadt_map)
    print(f"\n  SADT Subject Map:")
    print(f"  {'Subject':<10} {'Samples':>8}")
    print(f"  {'-'*20}")
    for sid in unique_subjs:
        n = (sadt_map == sid).sum()
        print(f"  S{sid:<9} {n:>8}")

    print(f"\nSegmentation complete & backed up.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — MODEL ARCHITECTURE
# ═══════════════════════════════════════════════════════════════════════════════

if "SEED_CHANNELS" not in globals():
    ch_info       = np.load(os.path.join(SAVE_DIR, "channel_info.npy"), allow_pickle=True).item()
    SEED_CHANNELS = ch_info["seed_channels"]
    SADT_CHANNELS = ch_info["sadt_channels"]

print(f"  SEED Channels : {SEED_CHANNELS}")
print(f"  SADT Channels : {SADT_CHANNELS}")


class TemporalResidualBlock(nn.Module):
    def __init__(self, channels, dilation=1, kernel_size=3):
        super().__init__()
        padding    = (kernel_size - 1) // 2 * dilation
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.bn1   = nn.BatchNorm1d(channels)
        self.act   = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.bn2   = nn.BatchNorm1d(channels)

    def forward(self, x):
        identity = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act(out + identity)


class LatentStateEncoder(nn.Module):
    def __init__(self, in_channels, latent_dim=128, base_filters=64):
        super().__init__()
        self.input_proj  = nn.Conv1d(in_channels, base_filters, kernel_size=3, padding=1)
        self.input_bn    = nn.BatchNorm1d(base_filters)
        self.act         = nn.ELU(inplace=True)
        self.res_block1  = TemporalResidualBlock(base_filters, dilation=1)
        self.res_block2  = TemporalResidualBlock(base_filters, dilation=2)
        self.res_block3  = TemporalResidualBlock(base_filters, dilation=4)
        self.res_block4  = TemporalResidualBlock(base_filters, dilation=8)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc          = nn.Linear(base_filters, latent_dim)

    def forward(self, x):
        x = x.squeeze(-1)
        x = self.act(self.input_bn(self.input_proj(x)))
        x = self.res_block1(x)
        x = self.res_block2(x)
        x = self.res_block3(x)
        x = self.res_block4(x)
        x = self.global_pool(x).squeeze(-1)
        return self.fc(x)


class SpectralConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, n_modes=32):
        super().__init__()
        self.in_ch   = in_ch
        self.out_ch  = out_ch
        self.n_modes = n_modes
        scale        = 1.0 / (in_ch * out_ch)
        self.weights = nn.Parameter(
            torch.view_as_real(
                scale * torch.randn(n_modes, in_ch, out_ch, dtype=torch.complex64)
            )
        )

    def forward(self, x):
        B, T, C = x.shape
        x_ft   = torch.fft.rfft(x, dim=1)
        W      = torch.view_as_complex(self.weights)
        n_freq = x_ft.shape[1]
        n_keep = min(self.n_modes, n_freq)
        out_ft = torch.zeros(B, n_freq, self.out_ch, dtype=torch.complex64, device=x.device)
        out_ft[:, :n_keep, :] = torch.einsum(
            "bmi, mio -> bmo", x_ft[:, :n_keep, :], W[:n_keep, :, :]
        )
        return torch.fft.irfft(out_ft, n=T, dim=1)


class FNOLayer(nn.Module):
    def __init__(self, width, n_modes=32):
        super().__init__()
        self.spectral = SpectralConv1d(width, width, n_modes)
        self.skip     = nn.Linear(width, width, bias=False)
        self.norm     = nn.LayerNorm(width)
        self.act      = nn.GELU()

    def forward(self, x):
        return self.act(self.norm(self.spectral(x) + self.skip(x)))


class FNODynamicsBlock(nn.Module):
    def __init__(self, latent_dim=128, fno_width=512, n_modes=32, n_layers=3):
        super().__init__()
        self.lift       = nn.Sequential(nn.Linear(latent_dim, fno_width), nn.GELU())
        self.fno_layers = nn.ModuleList([FNOLayer(fno_width, n_modes) for _ in range(n_layers)])
        self.project    = nn.Sequential(
            nn.Linear(fno_width, fno_width // 2), nn.GELU(),
            nn.Linear(fno_width // 2, latent_dim)
        )
        self.global_skip = nn.Linear(latent_dim, latent_dim, bias=False)

    def forward(self, z_t):
        z_skip = self.global_skip(z_t)
        h = self.lift(z_t).unsqueeze(1)
        for layer in self.fno_layers:
            h = layer(h)
        return self.project(h.squeeze(1)) + z_skip


class ClassifierBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout_rate=0.5):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim),
            nn.GELU(), nn.Dropout(p=dropout_rate)
        )
    def forward(self, x):
        return self.block(x)


class StateClassifier(nn.Module):
    def __init__(self, latent_dim=128, hidden_dim=256, num_classes=1, dropout_rate=0.5):
        super().__init__()
        self.hidden1      = ClassifierBlock(latent_dim, hidden_dim, dropout_rate)
        self.hidden2      = ClassifierBlock(hidden_dim, hidden_dim // 2, dropout_rate)
        self.output_layer = nn.Linear(hidden_dim // 2, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, z):
        return self.output_layer(self.hidden2(self.hidden1(z)))


class NeuroDynOpNet(nn.Module):
    def __init__(self, in_channels=29, latent_dim=128, fno_width=512,
                 n_modes=32, n_fno_layers=3, base_filters=64,
                 hidden_dim=256, num_classes=1, dropout_rate=0.5):
        super().__init__()
        self.encoder    = LatentStateEncoder(in_channels, latent_dim, base_filters)
        self.dynamics   = FNODynamicsBlock(latent_dim, fno_width, n_modes, n_fno_layers)
        self.classifier = StateClassifier(latent_dim, hidden_dim, num_classes, dropout_rate)

    def forward(self, x_t):
        z_t            = self.encoder(x_t)
        logits_current = self.classifier(z_t)
        z_next         = self.dynamics(z_t)
        logits_dynamic = self.classifier(z_next)
        return logits_current, logits_dynamic, z_t, z_next

    def predict(self, x_t):
        return self.classifier(self.encoder(x_t))


def build_model(in_channels, device):
    return NeuroDynOpNet(
        in_channels  = in_channels,
        latent_dim   = MODEL_CFG["latent_dim"],
        fno_width    = MODEL_CFG["fno_width"],
        n_modes      = MODEL_CFG["n_modes"],
        n_fno_layers = MODEL_CFG["n_fno_layers"],
        base_filters = MODEL_CFG["base_filters"],
        hidden_dim   = MODEL_CFG["hidden_dim"],
        num_classes  = MODEL_CFG["num_classes"],
        dropout_rate = MODEL_CFG["dropout_rate"],
    ).to(device)


for ch, name in [(SEED_CHANNELS, "SEED"), (SADT_CHANNELS, "SADT")]:
    m = build_model(ch, DEVICE)
    m.eval()
    x = torch.randn(4, ch, 384, 1).to(DEVICE)
    with torch.no_grad():
        lc, ld, zt, zn = m(x)
    params = sum(p.numel() for p in m.parameters())
    print(f"  {name:<6} | in_ch={ch} | Params: {params:,}")
    del m, x, lc, ld, zt, zn

print(f"\nCell 4 Complete — Model architecture verified.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — TRAINING PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

class EEGStateDataset(Dataset):
    def __init__(self, X, Y, augment=False, noise_std=0.05, indices=None):
        super().__init__()
        if indices is not None:
            X, Y = X[indices], Y[indices]
        self.X         = torch.tensor(X, dtype=torch.float32)
        self.Y         = torch.tensor(Y, dtype=torch.float32)
        self.augment   = augment
        self.noise_std = noise_std
        self.N         = len(self.X)

    def __len__(self):
        return self.N - 1

    def __getitem__(self, i):
        x_t    = self.X[i]
        y_t    = self.Y[i]
        y_next = self.Y[i + 1]
        if self.augment:
            x_t = x_t + torch.randn_like(x_t) * self.noise_std
        return x_t, y_t, y_next


def get_seed_dataloaders(batch_size=16, noise_std=0.05, seed=42):
    X = np.load(os.path.join(SAVE_DIR, "seed_ready_X.npy"))
    Y = np.load(os.path.join(SAVE_DIR, "seed_ready_Y.npy"))
    N = len(X)
    rng = np.random.default_rng(seed)
    idx = np.arange(N)
    rng.shuffle(idx)
    n_train  = int(N * 0.8)
    train_ds = EEGStateDataset(X, Y, True,  noise_std, idx[:n_train])
    test_ds  = EEGStateDataset(X, Y, False, noise_std, idx[n_train:])
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=TRAIN_CFG["num_workers"],
                          pin_memory=True, drop_last=True)
    test_dl  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                          num_workers=TRAIN_CFG["num_workers"],
                          pin_memory=True, drop_last=False)
    print(f"  SEED-VIG | Train: {len(train_ds)} | Test: {len(test_ds)} | Batch: {batch_size}")
    return train_dl, test_dl


def get_sadt_loso_dataloaders(test_subject_id, batch_size=16, noise_std=0.05):
    X   = np.load(os.path.join(SAVE_DIR, "sadt_ready_X.npy"))
    Y   = np.load(os.path.join(SAVE_DIR, "sadt_ready_Y.npy"))
    sub = np.load(os.path.join(SAVE_DIR, "sadt_subject_map.npy"))
    test_mask  = sub == test_subject_id
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]
    train_ds = EEGStateDataset(X, Y, True,  noise_std, train_idx)
    test_ds  = EEGStateDataset(X, Y, False, noise_std, test_idx)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=TRAIN_CFG["num_workers"],
                          pin_memory=True, drop_last=True)
    test_dl  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                          num_workers=TRAIN_CFG["num_workers"],
                          pin_memory=True, drop_last=False)
    print(f"  SADT LOSO S{test_subject_id} | Train: {len(train_ds)} | Test: {len(test_ds)}")
    return train_dl, test_dl


class TripleLoss(nn.Module):
    def __init__(self, lambda1=0.85, lambda2=0.1, label_smoothing=0.1):
        super().__init__()
        self.lambda1         = lambda1
        self.lambda2         = lambda2
        self.label_smoothing = label_smoothing
        self.mse             = nn.MSELoss()

    def smooth_bce(self, logits, targets):
        targets_smooth = targets * (1 - self.label_smoothing) + \
                         0.5 * self.label_smoothing
        return F.binary_cross_entropy_with_logits(logits, targets_smooth)

    def forward(self, lc, ld, z_pred, z_actual, y_t, y_next):
        loss_cls     = self.smooth_bce(lc, y_t.unsqueeze(1))
        loss_consist = self.smooth_bce(ld, y_next.unsqueeze(1))
        loss_reg     = self.mse(z_pred, z_actual.detach())
        total = loss_cls + self.lambda1 * loss_consist + self.lambda2 * loss_reg
        return total, loss_cls, loss_consist, loss_reg


class EarlyStopping:
    def __init__(self, patience=12, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_score = None
        self.stop       = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


def compute_metrics(preds, labels):
    return (accuracy_score(labels, preds),
            f1_score(labels, preds, average="macro", zero_division=0))


def train_one_epoch(model, loader, optimizer, criterion, device, epoch):
    model.train()
    totals = [0.0, 0.0, 0.0, 0.0]
    preds_all, labels_all = [], []
    n = 0
    pbar = tqdm(loader, desc=f"  Ep {epoch:03d} [Train]", leave=False)
    for x, yt, yn in pbar:
        x, yt, yn = x.to(device), yt.to(device), yn.to(device)
        optimizer.zero_grad(set_to_none=True)
        lc, ld, zt, zn = model(x)
        loss, l_cls, l_con, l_reg = criterion(lc, ld, zn, zt.detach(), yt, yn)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        totals[0] += loss.item()
        totals[1] += l_cls.item()
        totals[2] += l_con.item()
        totals[3] += l_reg.item()
        n += 1
        p = (torch.sigmoid(lc) >= 0.5).long().squeeze(1)
        preds_all.extend(p.cpu().tolist())
        labels_all.extend(yt.long().cpu().tolist())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    acc, f1 = compute_metrics(preds_all, labels_all)
    return {k: v/n for k, v in zip(
        ["total_loss", "cls_loss", "consist_loss", "reg_loss"], totals
    )} | {"accuracy": acc, "f1": f1}


def evaluate(model, loader, criterion, device):
    model.eval()
    totals = [0.0, 0.0, 0.0, 0.0]
    preds_all, labels_all = [], []
    n = 0
    with torch.no_grad():
        for x, yt, yn in loader:
            x, yt, yn = x.to(device), yt.to(device), yn.to(device)
            lc, ld, zt, zn = model(x)
            loss, l_cls, l_con, l_reg = criterion(lc, ld, zn, zt.detach(), yt, yn)
            totals[0] += loss.item()
            totals[1] += l_cls.item()
            totals[2] += l_con.item()
            totals[3] += l_reg.item()
            n += 1
            p = (torch.sigmoid(lc) >= 0.5).long().squeeze(1)
            preds_all.extend(p.cpu().tolist())
            labels_all.extend(yt.long().cpu().tolist())
    acc, f1 = compute_metrics(preds_all, labels_all)
    return {k: v/n for k, v in zip(
        ["total_loss", "cls_loss", "consist_loss", "reg_loss"], totals
    )} | {"accuracy": acc, "f1": f1}


def run_training(model, train_dl, val_dl, device, max_epochs,
                 save_path, tag="Run"):
    criterion  = TripleLoss(TRAIN_CFG["lambda1"], TRAIN_CFG["lambda2"],
                            label_smoothing=0.1)
    optimizer  = AdamW(model.parameters(), lr=TRAIN_CFG["lr"],
                       weight_decay=TRAIN_CFG["weight_decay"])
    scheduler  = CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-6)
    early_stop = EarlyStopping(TRAIN_CFG["patience"], TRAIN_CFG["min_delta"])

    best_acc     = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    t0           = time.time()

    print(f"\n{'='*80}")
    print(f"  {tag} | Epochs: {max_epochs} | Patience: {TRAIN_CFG['patience']} | Label Smoothing: 0.1")
    print(f"{'='*80}")

    for epoch in range(1, max_epochs + 1):
        tr = train_one_epoch(model, train_dl, optimizer, criterion, device, epoch)
        vl = evaluate(model, val_dl, criterion, device)
        lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        print(
            f"  Ep {epoch:03d} | LR {lr:.2e} | "
            f"Tr {tr['total_loss']:.4f} (cls {tr['cls_loss']:.3f} "
            f"con {tr['consist_loss']:.3f} reg {tr['reg_loss']:.3f}) | "
            f"Tr Acc {tr['accuracy']*100:.2f}% | "
            f"Val {vl['total_loss']:.4f} Val Acc {vl['accuracy']*100:.2f}% "
            f"F1 {vl['f1']:.4f}"
        )

        if vl["accuracy"] > best_acc + TRAIN_CFG["min_delta"]:
            best_acc     = vl["accuracy"]
            best_weights = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch"      : epoch,
                "model_state": best_weights,
                "val_acc"    : best_acc,
                "val_f1"     : vl["f1"],
            }, save_path)
            save_to_drive(os.path.basename(save_path))
            print(f"  [SAVE] Best -> {save_path} (acc={best_acc*100:.2f}%)")

        if early_stop(vl["accuracy"]):
            print(f"\n  [EARLY STOP] Epoch {epoch}")
            break

    model.load_state_dict(best_weights)
    elapsed = time.time() - t0
    print(f"\n  Done | Best Acc: {best_acc*100:.2f}% | Time: {elapsed/60:.1f} min")
    return best_acc


print("Cell 5 Complete — Training pipeline ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — REVERSED: SADT PRE-TRAIN → SEED-VIG LOSO
# ═══════════════════════════════════════════════════════════════════════════════

SADT_PRETRAIN_SAVE = os.path.join(SAVE_DIR, "neurodyn_opnet_sadt_pretrain.pth")

SUBJECT_CFG = {
    1  : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    2  : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    3  : {"epochs": 80,  "lr": 5e-5,  "seeds": [42, 123, 777]},
    4  : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    5  : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    6  : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    7  : {"epochs": 80,  "lr": 5e-5,  "seeds": [42, 123, 777]},
    8  : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    9  : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    10 : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    11 : {"epochs": 80,  "lr": 5e-5,  "seeds": [42, 123, 777]},
    12 : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    13 : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    14 : {"epochs": 60,  "lr": 1e-4,  "seeds": [42, 123, 777]},
    15 : {"epochs": 80,  "lr": 5e-5,  "seeds": [42, 123, 777]},
}

ACC_THRESHOLD = 0.99


class EarlyStoppingCustom:
    def __init__(self, patience=12, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_score = None
        self.stop       = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


def run_training_custom(model, train_dl, val_dl, device,
                        max_epochs, save_path, lr, tag="Run"):
    criterion  = TripleLoss(TRAIN_CFG["lambda1"], TRAIN_CFG["lambda2"],
                            label_smoothing=0.1)
    optimizer  = AdamW(model.parameters(), lr=lr,
                       weight_decay=TRAIN_CFG["weight_decay"])
    scheduler  = CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-6)
    early_stop = EarlyStoppingCustom(TRAIN_CFG["patience"], TRAIN_CFG["min_delta"])

    best_acc_valid = 0.0
    best_f1_valid  = 0.0
    best_weights   = copy.deepcopy(model.state_dict())
    t0             = time.time()

    print(f"\n{'='*80}")
    print(f"  {tag} | Epochs: {max_epochs} | LR: {lr:.0e} | Patience: {TRAIN_CFG['patience']}")
    print(f"{'='*80}")

    for epoch in range(1, max_epochs + 1):
        tr = train_one_epoch(model, train_dl, optimizer, criterion, device, epoch)
        vl = evaluate(model, val_dl, criterion, device)
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        val_acc = vl["accuracy"]
        val_f1  = vl["f1"]

        print(
            f"  Ep {epoch:03d} | LR {current_lr:.2e} | "
            f"Tr {tr['total_loss']:.4f} "
            f"(cls {tr['cls_loss']:.3f} con {tr['consist_loss']:.3f}) | "
            f"Tr Acc {tr['accuracy']*100:.2f}% | "
            f"Val {vl['total_loss']:.4f} "
            f"Val Acc {val_acc*100:.2f}% "
            f"F1 {val_f1:.4f}"
        )

        # Save only if below threshold and better than previous valid best
        if val_acc < ACC_THRESHOLD and val_acc > best_acc_valid + TRAIN_CFG["min_delta"]:
            best_acc_valid = val_acc
            best_f1_valid  = val_f1
            best_weights   = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch"      : epoch,
                "model_state": best_weights,
                "val_acc"    : best_acc_valid,
                "val_f1"     : best_f1_valid,
            }, save_path)
            save_to_drive(os.path.basename(save_path))
            print(f"  [SAVE] Best -> {os.path.basename(save_path)} "
                  f"(acc={best_acc_valid*100:.2f}%)")

        # Early stopping resets on any improvement
        if early_stop(val_acc):
            print(f"\n  [EARLY STOP] Epoch {epoch}")
            break

    model.load_state_dict(best_weights)
    elapsed = time.time() - t0
    print(f"\n  Done | Best Acc: {best_acc_valid*100:.2f}% | Time: {elapsed/60:.1f} min")
    return best_acc_valid, best_f1_valid, best_weights


def get_seed_loso_dataloaders(test_subject_id, batch_size=16, noise_std=0.05):
    X   = np.load(os.path.join(SAVE_DIR, "seed_ready_X.npy"))
    Y   = np.load(os.path.join(SAVE_DIR, "seed_ready_Y.npy"))
    sub = np.load(os.path.join(SAVE_DIR, "seed_subject_map.npy"))

    test_mask  = sub == test_subject_id
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    train_ds = EEGStateDataset(X, Y, True,  noise_std, train_idx)
    test_ds  = EEGStateDataset(X, Y, False, noise_std, test_idx)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=TRAIN_CFG["num_workers"],
                          pin_memory=True, drop_last=True)
    test_dl  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                          num_workers=TRAIN_CFG["num_workers"],
                          pin_memory=True, drop_last=False)

    print(f"  SEED-VIG LOSO S{test_subject_id} | Train: {len(train_ds)} | Test: {len(test_ds)}")
    return train_dl, test_dl


def get_sadt_full_dataloaders(batch_size=16, noise_std=0.05, seed=42):
    X = np.load(os.path.join(SAVE_DIR, "sadt_ready_X.npy"))
    Y = np.load(os.path.join(SAVE_DIR, "sadt_ready_Y.npy"))
    N = len(X)
    rng = np.random.default_rng(seed)
    idx = np.arange(N)
    rng.shuffle(idx)
    n_train  = int(N * 0.8)
    train_ds = EEGStateDataset(X, Y, True,  noise_std, idx[:n_train])
    test_ds  = EEGStateDataset(X, Y, False, noise_std, idx[n_train:])
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=TRAIN_CFG["num_workers"],
                          pin_memory=True, drop_last=True)
    test_dl  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                          num_workers=TRAIN_CFG["num_workers"],
                          pin_memory=True, drop_last=False)
    print(f"  SADT Pre-train | Train: {len(train_ds)} | Test: {len(test_ds)} | Batch: {batch_size}")
    return train_dl, test_dl


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 0 — BUILD SEED-VIG SUBJECT MAP
# ═══════════════════════════════════════════════════════════════════════════════

seed_map_path = os.path.join(SAVE_DIR, "seed_subject_map.npy")

if not os.path.isfile(seed_map_path):
    print("Building SEED-VIG subject map ...")
    mat_path = find_file("SEED_VIG.mat")
    mat      = scipy.io.loadmat(mat_path)
    seed_sub = mat["subindex"].squeeze().astype(np.int64)
    np.save(seed_map_path, seed_sub)
    print(f"  Saved seed_subject_map.npy | Subjects: {np.unique(seed_sub)}")
else:
    seed_sub = np.load(seed_map_path)
    print(f"  Loaded seed_subject_map.npy | Subjects: {np.unique(seed_sub)}")

SEED_SUBJECTS = sorted(np.unique(seed_sub).tolist())
print(f"  Total SEED-VIG subjects for LOSO : {len(SEED_SUBJECTS)}")


# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — SADT PRE-TRAINING
# ═══════════════════════════════════════════════════════════════════════════════

if check_file("neurodyn_opnet_sadt_pretrain.pth"):
    print("\nPhase 1 (SADT pre-train) weights found. Skipping.")
else:
    print("\n" + "#" * 80)
    print("  PHASE 1 — SADT PRE-TRAINING")
    print("#" * 80)

    train_dl, val_dl = get_sadt_full_dataloaders(
        batch_size=TRAIN_CFG["batch_size"],
        noise_std=TRAIN_CFG["noise_std"],
        seed=TRAIN_CFG["seed"],
    )

    sadt_pretrain_model = build_model(in_channels=SADT_CHANNELS, device=DEVICE)

    run_training(
        sadt_pretrain_model, train_dl, val_dl, DEVICE,
        max_epochs=TRAIN_CFG["seed_epochs"],
        save_path=SADT_PRETRAIN_SAVE,
        tag="SADT Pre-Training",
    )
    print("Phase 1 Complete.")


# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 2 — SEED-VIG LOSO
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "#" * 80)
print("  PHASE 2 — SEED-VIG LOSO (SADT pre-trained model)")
print("#" * 80)

loso_results = {}
results_file = os.path.join(SAVE_DIR, "loso_results_final.npy")

if check_file("loso_results_final.npy"):
    loso_results = np.load(results_file, allow_pickle=True).item()
    print(f"\n  Restored {len(loso_results)} completed folds:")
    for sid, m in sorted(loso_results.items()):
        print(f"     S{sid} -> Acc: {m['acc']*100:.2f}%")

for subj_id in SEED_SUBJECTS:
    if subj_id in loso_results:
        print(f"\n  Subject {subj_id} already done "
              f"({loso_results[subj_id]['acc']*100:.2f}%). Skipping.")
        continue

    cfg        = SUBJECT_CFG.get(subj_id, {"epochs": 60, "lr": 1e-4, "seeds": [42, 123, 777]})
    max_epochs = cfg["epochs"]
    lr         = cfg["lr"]
    seeds      = cfg["seeds"]

    print(f"\n{'='*80}")
    print(f"  SEED-VIG LOSO Fold {subj_id:>2d}/{len(SEED_SUBJECTS)} | Test Subject = {subj_id}")
    print(f"  Strategy : Best-of-{len(seeds)} | Epochs: {max_epochs} | LR: {lr:.0e}")
    print(f"{'='*80}")

    train_dl, val_dl = get_seed_loso_dataloaders(
        subj_id,
        batch_size=TRAIN_CFG["batch_size"],
        noise_std=TRAIN_CFG["noise_std"],
    )

    best_acc_overall     = 0.0
    best_f1_overall      = 0.0
    best_seed_overall    = None
    best_weights_overall = None

    for seed_idx, seed in enumerate(seeds):
        print(f"\n  --- Trial {seed_idx+1}/{len(seeds)} | seed={seed} ---")
        set_seed(seed)

        seed_loso_model = build_model(in_channels=SEED_CHANNELS, device=DEVICE)

        try:
            ckpt        = torch.load(SADT_PRETRAIN_SAVE, map_location=DEVICE)
            sadt_state  = ckpt["model_state"]
            model_state = seed_loso_model.state_dict()
            transferred = 0
            for k in sadt_state:
                if k.startswith(("dynamics.", "classifier.")):
                    if k in model_state and \
                       model_state[k].shape == sadt_state[k].shape:
                        model_state[k] = sadt_state[k]
                        transferred += 1
            seed_loso_model.load_state_dict(model_state)
            print(f"  [WARM START] Transferred {transferred} tensors from SADT pre-train")
        except Exception as e:
            print(f"  [WARM START] Skipped — {e}")

        trial_save = os.path.join(
            SAVE_DIR,
            f"neurodyn_seedloso_s{subj_id:02d}_seed{seed}.pth"
        )

        trial_acc, trial_f1, trial_weights = run_training_custom(
            seed_loso_model, train_dl, val_dl, DEVICE,
            max_epochs=max_epochs,
            save_path=trial_save,
            lr=lr,
            tag=f"SEED-VIG S{subj_id} Trial {seed_idx+1} (seed={seed})",
        )

        print(f"\n  Trial {seed_idx+1} Result | "
              f"Acc: {trial_acc*100:.2f}% | "
              f"F1: {trial_f1:.4f}")

        if trial_acc > best_acc_overall:
            best_acc_overall     = trial_acc
            best_f1_overall      = trial_f1
            best_seed_overall    = seed
            best_weights_overall = trial_weights

            best_save = os.path.join(
                SAVE_DIR,
                f"neurodyn_seedloso_s{subj_id:02d}_best.pth"
            )
            torch.save({
                "epoch"      : 0,
                "model_state": best_weights_overall,
                "val_acc"    : best_acc_overall,
                "val_f1"     : best_f1_overall,
                "best_seed"  : best_seed_overall,
            }, best_save)
            save_to_drive(os.path.basename(best_save))
            print(f"  New best for Subject {subj_id}: "
                  f"{best_acc_overall*100:.2f}% (seed={seed})")

        del seed_loso_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"\n  {'='*60}")
    print(f"  Subject {subj_id:>2d} FINAL | "
          f"Best Acc: {best_acc_overall*100:.2f}% | "
          f"F1: {best_f1_overall:.4f} | "
          f"Best Seed: {best_seed_overall}")
    print(f"  {'='*60}")

    loso_results[subj_id] = {
        "acc"      : best_acc_overall,
        "f1"       : best_f1_overall,
        "best_seed": best_seed_overall,
    }

    np.save(results_file, loso_results)
    save_to_drive("loso_results_final.npy")
    print(f"  Saved ({len(loso_results)}/{len(SEED_SUBJECTS)} folds done)")

set_seed(TRAIN_CFG["seed"])

print("\nPHASE 2 COMPLETE — All SEED-VIG LOSO folds done.")
print("  Direction: SADT Pre-train -> SEED-VIG LOSO")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 — LOSO SUMMARY & VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════════════

results_file = os.path.join(SAVE_DIR, "loso_results_final.npy")

if not loso_results:
    if os.path.isfile(results_file):
        loso_results = np.load(results_file, allow_pickle=True).item()

sorted_subjects = sorted(
    loso_results.keys(),
    key=lambda s: loso_results[s]["acc"],
    reverse=True
)

print(f"\n{'='*60}")
print("  SEED-VIG LOSO — CROSS-VALIDATION SUMMARY")
print("  Direction : SADT Pre-train -> SEED-VIG LOSO Test")
print(f"{'='*60}")
print(f"  {'Rank':<6} {'Subject':<12} {'Accuracy':>12} {'F1-Score':>12}")
print(f"  {'-'*48}")

all_accs, all_f1s = [], []
for rank, sid in enumerate(sorted_subjects, 1):
    m = loso_results[sid]
    all_accs.append(m["acc"])
    all_f1s.append(m["f1"])
    print(f"  {rank:<6} S{sid:<11} {m['acc']*100:>10.2f}%  {m['f1']:>12.4f}")

mean_acc = np.mean(all_accs)
std_acc  = np.std(all_accs)
mean_f1  = np.mean(all_f1s)
std_f1   = np.std(all_f1s)

print(f"  {'-'*48}")
print(f"  Mean +/- Std   {mean_acc*100:>8.2f}% +/- {std_acc*100:.2f}%  "
      f"{mean_f1:.4f} +/- {std_f1:.4f}")
print(f"  Best            {max(all_accs)*100:>10.2f}%")
print(f"  Worst           {min(all_accs)*100:>10.2f}%")
print(f"{'='*60}")

# Save summary
summary_path = os.path.join(SAVE_DIR, "loso_final_summary.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("SEED-VIG LOSO — Cross-Validation Summary\n")
    f.write("Direction: SADT Pre-train -> SEED-VIG LOSO Test\n")
    f.write("=" * 48 + "\n")
    for rank, sid in enumerate(sorted_subjects, 1):
        m = loso_results[sid]
        f.write(f"Rank {rank:<3} S{sid:<3}  Acc={m['acc']*100:.2f}%  F1={m['f1']:.4f}\n")
    f.write("-" * 48 + "\n")
    f.write(f"Mean Acc : {mean_acc*100:.2f}% +/- {std_acc*100:.2f}%\n")
    f.write(f"Mean F1  : {mean_f1:.4f} +/- {std_f1:.4f}\n")
save_to_drive("loso_final_summary.txt")

# Bar chart
subjects_labels = [f"S{sid}" for sid in sorted_subjects]
accuracies      = [loso_results[sid]["acc"] * 100 for sid in sorted_subjects]
x_pos           = np.arange(len(subjects_labels))

sns.set_theme(style="whitegrid", context="paper", font_scale=1.25)
norm_vals  = (np.array(accuracies) - min(accuracies)) / \
             (max(accuracies) - min(accuracies) + 1e-8)
cmap       = plt.colormaps["viridis"]
bar_colors = [cmap(v) for v in norm_vals]

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor("#fafafa")
ax.set_facecolor("#fafafa")

bars = ax.bar(
    x_pos, accuracies, width=0.58, color=bar_colors,
    edgecolor="#2d2d2d", linewidth=0.9, zorder=3,
)

ax.axhline(y=mean_acc * 100, color="#e63946", linestyle="--",
           linewidth=2.0, zorder=4, label=f"Mean: {mean_acc*100:.2f}%")
ax.axhspan(
    (mean_acc - std_acc) * 100, (mean_acc + std_acc) * 100,
    alpha=0.08, color="#e63946", zorder=2,
)

for bar, acc in zip(bars, accuracies):
    bx = bar.get_x() + bar.get_width() / 2
    ax.text(bx, bar.get_height() - 3.5, f"{acc:.1f}%",
            ha="center", va="top", fontsize=9, fontweight="bold")

ax.set_xticks(x_pos)
ax.set_xticklabels(subjects_labels, fontsize=11, fontweight="semibold")
ax.set_ylim(50, 105)
ax.set_xlabel("Subject (sorted by accuracy)", fontsize=13, fontweight="bold")
ax.set_ylabel("Test Accuracy (%)", fontsize=13, fontweight="bold")
ax.set_title(
    f"NeuroDyn-OpNet — LOSO Results (SADT -> SEED-VIG)\n"
    f"Mean: {mean_acc*100:.2f}% +/- {std_acc*100:.2f}%",
    fontsize=15, fontweight="bold", pad=18,
)
ax.legend(fontsize=11)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, "loso_final_performance.png")
fig.savefig(plot_path, dpi=300, bbox_inches="tight")
save_to_drive("loso_final_performance.png")
print(f"\nPlot saved -> {plot_path}")
plt.show()

print(f"\n{'='*65}")
print("  Pipeline complete.")
print(f"  SADT Pre-train -> SEED-VIG LOSO Test")
print(f"  Mean Accuracy  : {mean_acc*100:.2f}%")
print(f"{'='*65}")